In [ ]:
import os
import random
import numpy as np
import pandas as pd
import mne

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


# -----------------------------
# Config
# -----------------------------
DATA_DIR = "data"

SUBJECTS = range(1, 10)

SEED = 42
BATCH_SIZE = 32
N_EPOCHS = 30
LR = 1e-3

# Epoch relative to cue onset.
# Event 769/770/771/772 = cue onset for each MI class.
TMIN = 0.0
TMAX = 4.0

CLASS_NAMES = ["left_hand", "right_hand", "feet", "tongue"]

EVENT_ID = {
    "left_hand": 769,
    "right_hand": 770,
    "feet": 771,
    "tongue": 772,
}

EOG_CHANNELS = ["EOG-left", "EOG-central", "EOG-right"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# -----------------------------
# Event parser for BCI IV 2a
# -----------------------------
def bci2a_event_parser(description):
    """
    Keep only the four motor imagery cue events:
    769 = left hand
    770 = right hand
    771 = feet
    772 = tongue
    """
    description = str(description).strip()

    if description in ["769", "770", "771", "772"]:
        return int(description)

    return None


# -----------------------------
# Dataset loading + preprocessing
# -----------------------------
def load_subject_training_file(subject_id):
    file_path = os.path.join(DATA_DIR, f"A{subject_id:02d}T.gdf")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    raw = mne.io.read_raw_gdf(
        file_path,
        eog=EOG_CHANNELS,
        preload=True,
        verbose=False
    )

    # Drop EOG channels because we only want EEG for classification.
    existing_eog = [ch for ch in EOG_CHANNELS if ch in raw.ch_names]
    if len(existing_eog) > 0:
        raw.drop_channels(existing_eog)

    # Keep EEG channels only.
    raw.pick_types(eeg=True, eog=False)

    # Some BCI IV 2a GDF files contain NaN separators between runs.
    # Replace NaNs before filtering.
    raw._data[np.isnan(raw._data)] = 0.0

    # Basic motor imagery band.
    raw.filter(8, 30, fir_design="firwin", verbose=False)

    # Extract cue events.
    events, _ = mne.events_from_annotations(
        raw,
        event_id=bci2a_event_parser,
        verbose=False
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id=EVENT_ID,
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data().astype(np.float32)   # shape: trials, channels, samples
    y_codes = epochs.events[:, -1]             # 769, 770, 771, 772

    # Convert labels:
    # 769 -> 0, 770 -> 1, 771 -> 2, 772 -> 3
    y = (y_codes - 769).astype(np.int64)

    return X, y


# -----------------------------
# Simple ShallowConvNet-style model
# -----------------------------
class BasicShallowConvNet(nn.Module):
    def __init__(self, n_channels, n_samples, n_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            # Temporal convolution
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=(1, 25),
                padding=(0, 12),
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),

            # Spatial convolution across EEG channels
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=(n_channels, 1),
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ELU(),

            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.5),
            nn.Flatten()
        )

        # Automatically calculate classifier input size.
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            feature_dim = self.features(dummy).shape[1]

        self.classifier = nn.Linear(feature_dim, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# -----------------------------
# Train function
# -----------------------------
def train_model(model, train_loader):
    model.to(DEVICE)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(N_EPOCHS):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model


# -----------------------------
# Prediction function
# -----------------------------
def predict(model, loader):
    model.eval()

    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)

            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_true.extend(yb.numpy())

    return np.array(all_true), np.array(all_preds)


# -----------------------------
# Run one subject
# -----------------------------
def run_subject(subject_id):
    X, y = load_subject_training_file(subject_id)

    # Stratified split keeps the 4 classes balanced in train and test.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    # Normalize using training data only.
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True)
    std[std < 1e-6] = 1.0

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # PyTorch input shape: trials, 1, channels, samples
    X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test_t, y_test_t),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    n_channels = X_train.shape[1]
    n_samples = X_train.shape[2]

    model = BasicShallowConvNet(
        n_channels=n_channels,
        n_samples=n_samples,
        n_classes=4
    )

    model = train_model(model, train_loader)

    y_true, y_pred = predict(model, test_loader)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    result = {
        "subject": f"A{subject_id:02d}",
        "n_trials": len(X),
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
    }

    return result


# -----------------------------
# Run all subjects
# -----------------------------
def main():
    print(f"Using device: {DEVICE}")

    results = []

    for subject_id in SUBJECTS:
        print(f"Running subject A{subject_id:02d}...")

        try:
            result = run_subject(subject_id)
            results.append(result)

            print(
                f"A{subject_id:02d} | "
                f"acc={result['accuracy']:.4f}, "
                f"bal_acc={result['balanced_accuracy']:.4f}, "
                f"macro_f1={result['macro_f1']:.4f}"
            )

        except Exception as e:
            print(f"Subject A{subject_id:02d} failed: {e}")

    results_df = pd.DataFrame(results)

    print("\nPer-subject results:")
    print(results_df.round(4).to_string(index=False))

    print("\nMean results:")
    print(
        results_df[
            ["accuracy", "balanced_accuracy", "macro_f1"]
        ].mean().round(4)
    )

    print("\nStandard deviation:")
    print(
        results_df[
            ["accuracy", "balanced_accuracy", "macro_f1"]
        ].std().round(4)
    )

    results_df.to_csv("bci2a_basic_pytorch_results.csv", index=False)

    print("\nSaved results to: bci2a_basic_pytorch_results.csv")


if __name__ == "__main__":
    main()

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import mne

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import CSP


# -----------------------------
# Config
# -----------------------------
DATA_DIR = "data"
SUBJECTS = range(1, 10)

SEED = 42
BATCH_SIZE = 32
N_EPOCHS = 30
LR = 1e-3

TMIN = 0.0
TMAX = 4.0

CLASS_NAMES = ["left_hand", "right_hand", "feet", "tongue"]

EVENT_ID = {
    "left_hand": 769,
    "right_hand": 770,
    "feet": 771,
    "tongue": 772,
}

EOG_CHANNELS = ["EOG-left", "EOG-central", "EOG-right"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# -----------------------------
# Event parser
# -----------------------------
def bci2a_event_parser(description):
    """
    Keep only motor imagery cue events:
    769 = left hand
    770 = right hand
    771 = feet
    772 = tongue
    """
    description = str(description).strip()

    if description in ["769", "770", "771", "772"]:
        return int(description)

    return None


# -----------------------------
# Dataset loading + preprocessing
# -----------------------------
def load_subject_training_file(subject_id):
    file_path = os.path.join(DATA_DIR, f"A{subject_id:02d}T.gdf")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    raw = mne.io.read_raw_gdf(
        file_path,
        eog=EOG_CHANNELS,
        preload=True,
        verbose=False
    )

    existing_eog = [ch for ch in EOG_CHANNELS if ch in raw.ch_names]
    if len(existing_eog) > 0:
        raw.drop_channels(existing_eog)

    raw.pick_types(eeg=True, eog=False)

    # Some files contain NaN separators between runs.
    raw._data[np.isnan(raw._data)] = 0.0

    # Basic motor imagery band.
    raw.filter(8, 30, fir_design="firwin", verbose=False)

    events, _ = mne.events_from_annotations(
        raw,
        event_id=bci2a_event_parser,
        verbose=False
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id=EVENT_ID,
        tmin=TMIN,
        tmax=TMAX,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data().astype(np.float32)   # trials, channels, samples
    y_codes = epochs.events[:, -1]             # 769, 770, 771, 772

    y = (y_codes - 769).astype(np.int64)       # 0, 1, 2, 3

    return X, y


# -----------------------------
# Metrics
# -----------------------------
def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }


# ============================================================
# MODEL 1: CSP + LDA
# ============================================================
def train_eval_csp_lda(X_train, X_test, y_train, y_test):
    model = Pipeline([
        ("csp", CSP(
            n_components=8,
            reg="ledoit_wolf",
            log=True,
            norm_trace=False
        )),
        ("lda", LinearDiscriminantAnalysis())
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return calculate_metrics(y_test, y_pred)


# ============================================================
# MODEL 2: ShallowConvNet
# ============================================================
class ShallowConvNet(nn.Module):
    def __init__(self, n_channels, n_samples, n_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            # Temporal convolution
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=(1, 25),
                padding=(0, 12),
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ELU(),

            # Spatial convolution across EEG channels
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=(n_channels, 1),
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ELU(),

            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.5),
            nn.Flatten()
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            feature_dim = self.features(dummy).shape[1]

        self.classifier = nn.Linear(feature_dim, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def train_eval_shallowconvnet(X_train, X_test, y_train, y_test):
    # Normalize using train data only.
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True)
    std[std < 1e-6] = 1.0

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # PyTorch input shape:
    # trials, 1, channels, samples
    X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test_t, y_test_t),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    n_channels = X_train.shape[1]
    n_samples = X_train.shape[2]

    model = ShallowConvNet(
        n_channels=n_channels,
        n_samples=n_samples,
        n_classes=4
    ).to(DEVICE)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(N_EPOCHS):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()

    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)

            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_true.extend(yb.numpy())

    return calculate_metrics(np.array(all_true), np.array(all_preds))


# -----------------------------
# Run one subject
# -----------------------------
def run_subject(subject_id):
    X, y = load_subject_training_file(subject_id)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    subject_results = []

    # Model 1: CSP + LDA
    csp_results = train_eval_csp_lda(
        X_train,
        X_test,
        y_train,
        y_test
    )

    csp_results["subject"] = f"A{subject_id:02d}"
    csp_results["model"] = "CSP_LDA"
    csp_results["n_trials"] = len(X)
    csp_results["n_train"] = len(X_train)
    csp_results["n_test"] = len(X_test)

    subject_results.append(csp_results)

    # Model 2: ShallowConvNet
    shallow_results = train_eval_shallowconvnet(
        X_train,
        X_test,
        y_train,
        y_test
    )

    shallow_results["subject"] = f"A{subject_id:02d}"
    shallow_results["model"] = "ShallowConvNet"
    shallow_results["n_trials"] = len(X)
    shallow_results["n_train"] = len(X_train)
    shallow_results["n_test"] = len(X_test)

    subject_results.append(shallow_results)

    return subject_results


# -----------------------------
# Main
# -----------------------------
def main():
    print(f"Using device: {DEVICE}")

    all_results = []

    for subject_id in SUBJECTS:
        print(f"\nRunning subject A{subject_id:02d}...")

        try:
            subject_results = run_subject(subject_id)
            all_results.extend(subject_results)

            for result in subject_results:
                print(
                    f"{result['subject']} | "
                    f"{result['model']} | "
                    f"acc={result['accuracy']:.4f}, "
                    f"bal_acc={result['balanced_accuracy']:.4f}, "
                    f"macro_f1={result['macro_f1']:.4f}"
                )

        except Exception as e:
            print(f"Subject A{subject_id:02d} failed: {e}")

    results_df = pd.DataFrame(all_results)

    column_order = [
        "subject",
        "model",
        "n_trials",
        "n_train",
        "n_test",
        "accuracy",
        "balanced_accuracy",
        "macro_f1"
    ]

    results_df = results_df[column_order]

    print("\nPer-subject results:")
    print(results_df.round(4).to_string(index=False))

    print("\nMean results by model:")
    print(
        results_df
        .groupby("model")[["accuracy", "balanced_accuracy", "macro_f1"]]
        .mean()
        .round(4)
    )

    print("\nStandard deviation by model:")
    print(
        results_df
        .groupby("model")[["accuracy", "balanced_accuracy", "macro_f1"]]
        .std()
        .round(4)
    )

    results_df.to_csv("bci2a_two_baseline_results.csv", index=False)

    print("\nSaved results to: bci2a_two_baseline_results.csv")


if __name__ == "__main__":
    main()